# Clean

> Preprocessing and cleanup of raw diff ops before display.

In [ ]:
#| default_exp clean

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import re
from typing import List, Dict, Any

## Cleaning whitespace-only changes

Raw diffs often contain noisy whitespace-only changes (extra spaces, indent tweaks, newline shifts).
`clean_newlines` auto-accepts these so they don't clutter the UI, then merges consecutive ops of the same type.

In [ ]:
#| export
def clean_newlines(ops: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Auto-accept changes that are ONLY whitespace/newlines.
    
    Runs before grouping to ensure formatting changes don't noise up the diffs.
    After cleaning, merges consecutive ops of the same simple type."""
    is_whitespace = lambda t: t is not None and bool(re.match(r'^\s+$', t))
    cleaned = []

    for seg in ops:
        op_type = seg.get("type")
        text = seg.get("text", "")

        if op_type == 'add':
            if is_whitespace(text):
                cleaned.append({**seg, 'type': 'equal'})
            else:
                cleaned.append(seg)

        elif op_type == 'remove':
            if is_whitespace(text):
                continue  # drop whitespace-only removals
            else:
                cleaned.append(seg)

        elif op_type == 'replace':
            old_t = seg.get('old_text', '')
            new_t = seg.get('new_text', '')

            if is_whitespace(old_t) and is_whitespace(new_t):
                cleaned.append({'type': 'equal', 'text': new_t})
            elif is_whitespace(old_t) and not new_t:
                continue
            elif not old_t and is_whitespace(new_t):
                cleaned.append({'type': 'equal', 'text': new_t})
            else:
                cleaned.append(seg)
        else:
            cleaned.append(seg)

    # Merge consecutive ops of the same simple type
    if not cleaned:
        return []

    merged = []
    current = cleaned[0].copy()

    for next_seg in cleaned[1:]:
        if (current['type'] == next_seg['type'] and
            current['type'] in ['equal', 'add', 'remove'] and
            'text' in current and 'text' in next_seg):
            current['text'] += next_seg['text']
        else:
            merged.append(current)
            current = next_seg.copy()

    merged.append(current)
    return merged

In [ ]:
# whitespace-only add -> becomes equal
ops = [{"type": "equal", "text": "hello"}, {"type": "add", "text": "  "}]
result = clean_newlines(ops)
assert len(result) == 1  # merged into single equal
assert result[0] == {"type": "equal", "text": "hello  "}

In [ ]:
# whitespace-only remove -> dropped
ops = [{"type": "equal", "text": "hello"}, {"type": "remove", "text": "\n"}, {"type": "equal", "text": "world"}]
result = clean_newlines(ops)
assert len(result) == 1
assert result[0] == {"type": "equal", "text": "helloworld"}

In [ ]:
# whitespace replace (indent change) -> equal with new whitespace
ops = [{"type": "replace", "old_text": "  ", "new_text": "    "}]
result = clean_newlines(ops)
assert result == [{"type": "equal", "text": "    "}]

In [ ]:
# real changes pass through untouched
ops = [
    {"type": "equal", "text": "the "},
    {"type": "replace", "old_text": "old", "new_text": "new"},
    {"type": "equal", "text": " cat"},
]
result = clean_newlines(ops)
assert len(result) == 3
assert result[1] == {"type": "replace", "old_text": "old", "new_text": "new"}

In [ ]:
# empty input
assert clean_newlines([]) == []

## Extracting newlines into separate blocks

For the frontend to render line breaks correctly, we split ops at `\n` boundaries
so each newline is its own `equal` block. This makes it easy to map ops to visual lines.

In [ ]:
#| export
def extract_newlines_to_blocks(ops: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Extract all newlines into their own separate equal blocks.
    
    Rules:
    - Newlines in equal/add blocks -> separate equal blocks
    - Newlines in remove blocks -> dropped (not shown)
    - Newlines in replace blocks -> separate equal blocks (when symmetric)"""
    if not ops:
        return []

    result = []

    for op in ops:
        op_type = op["type"]

        if op_type == "replace":
            old_text = op.get("old_text", "")
            new_text = op.get("new_text", "")
            old_parts = old_text.split("\n")
            new_parts = new_text.split("\n")

            if len(old_parts) == len(new_parts):
                for i, (old_part, new_part) in enumerate(zip(old_parts, new_parts)):
                    if old_part or new_part:
                        result.append({"type": "replace", "old_text": old_part, "new_text": new_part})
                    if i < len(old_parts) - 1:
                        result.append({"type": "equal", "text": "\n"})
            else:
                # Asymmetric newlines - keep as-is
                result.append(op)

        elif op_type in ("equal", "add"):
            text = op.get("text", "")
            parts = text.split("\n")
            for i, part in enumerate(parts):
                if part:
                    result.append({"type": op_type, "text": part})
                if i < len(parts) - 1:
                    result.append({"type": "equal", "text": "\n"})

        elif op_type == "remove":
            text = op.get("text", "")
            parts = text.split("\n")
            for part in parts:
                if part:
                    result.append({"type": "remove", "text": part})

        else:
            result.append(op)

    return result

In [ ]:
# equal with newline splits into text + newline + text
ops = [{"type": "equal", "text": "hello\nworld"}]
result = extract_newlines_to_blocks(ops)
assert result == [
    {"type": "equal", "text": "hello"},
    {"type": "equal", "text": "\n"},
    {"type": "equal", "text": "world"},
]

In [ ]:
# add with newline splits similarly
ops = [{"type": "add", "text": "line1\nline2"}]
result = extract_newlines_to_blocks(ops)
assert result == [
    {"type": "add", "text": "line1"},
    {"type": "equal", "text": "\n"},
    {"type": "add", "text": "line2"},
]

In [ ]:
# remove with newline drops the newlines
ops = [{"type": "remove", "text": "old1\nold2"}]
result = extract_newlines_to_blocks(ops)
assert result == [
    {"type": "remove", "text": "old1"},
    {"type": "remove", "text": "old2"},
]

In [ ]:
# symmetric replace with newlines
ops = [{"type": "replace", "old_text": "a\nb", "new_text": "x\ny"}]
result = extract_newlines_to_blocks(ops)
assert result == [
    {"type": "replace", "old_text": "a", "new_text": "x"},
    {"type": "equal", "text": "\n"},
    {"type": "replace", "old_text": "b", "new_text": "y"},
]

In [ ]:
# asymmetric replace kept as-is
ops = [{"type": "replace", "old_text": "a", "new_text": "x\ny"}]
result = extract_newlines_to_blocks(ops)
assert len(result) == 1
assert result[0]["type"] == "replace"

In [ ]:
# empty input
assert extract_newlines_to_blocks([]) == []

## Adding accepted flags

Before handing ops to the frontend, we tag each with an `accepted` field.
Equal blocks are always accepted. Replace ops that just delete 1-3 spaces are auto-accepted too.
Everything else defaults to not accepted (pending user review).

In [ ]:
#| export
def add_accepted_flags(ops: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """Add `accepted` field to ops based on auto-accept rules.
    
    - equal blocks -> always accepted
    - replace that deletes 1-3 spaces (old=' '/' '/'   ', new='') -> convert to equal
    - add/remove -> not accepted (pending user review)"""
    if not ops:
        return []

    result = []
    for op in ops:
        op_type = op.get("type")

        if op_type == "equal":
            result.append({"type": "equal", "text": op.get("text", ""), "accepted": True})

        elif op_type == "replace":
            old_text = op.get("old_text", "")
            new_text = op.get("new_text", "")
            if len(new_text) == 0 and old_text in (' ', '  ', '   '):
                result.append({"type": "equal", "text": new_text, "accepted": True})
            else:
                result.append({"type": "replace", "old_text": old_text, "new_text": new_text, "accepted": False})

        elif op_type == "add":
            result.append({"type": "add", "text": op.get("text", ""), "accepted": False})

        elif op_type == "remove":
            result.append({"type": "remove", "text": op.get("text", ""), "accepted": False})

        else:
            result.append(op.copy())

    return result

In [ ]:
# equal -> accepted
result = add_accepted_flags([{"type": "equal", "text": "hello"}])
assert result == [{"type": "equal", "text": "hello", "accepted": True}]

In [ ]:
# add/remove -> not accepted
result = add_accepted_flags([
    {"type": "add", "text": "new"},
    {"type": "remove", "text": "old"},
])
assert result[0]["accepted"] == False
assert result[1]["accepted"] == False

In [ ]:
# replace deleting 1-3 spaces -> auto-accepted as equal
result = add_accepted_flags([{"type": "replace", "old_text": "  ", "new_text": ""}])
assert result == [{"type": "equal", "text": "", "accepted": True}]

In [ ]:
# real replace -> not accepted
result = add_accepted_flags([{"type": "replace", "old_text": "old", "new_text": "new"}])
assert result == [{"type": "replace", "old_text": "old", "new_text": "new", "accepted": False}]

In [ ]:
# empty
assert add_accepted_flags([]) == []

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()